In [1]:
from math import pi
# ngsolve stuff
from ngsolve import *
import numpy as np
import scipy.sparse as sp

# basic geometry features (for the background mesh)
from netgen.geom2d import SplineGeometry
from netgen.geom2d import CSG2d,Rectangle
from ngsolve.meshes import MakeStructured2DMesh, Make1DMesh, MakeQuadMesh

import matplotlib.pyplot as plt
from ngsolve.solvers import Newton
from ngsolve.webgui import Draw
import warnings
warnings.filterwarnings('ignore')

In [2]:
def logDensity(
nx = 20,
m = 2,
s0 = 3,
order = 1,
dt0 = 1e-2,
t = Parameter(0),
tstop = 4,
plot = 0,
cutoff = 1e-14,
quads = 0
):
    cutOff = 1e-14
    order = 1
    eps0 = 0 
    dim = 2
    dt = Parameter(dt0)
    
    k = 1/(m-1+2/dim)   
    xmin = -1
    xmax = 1
    L = xmax
    tend = tstop
    t0 = 1
    # reference solution
    rhoex = exp(-20*((x-0.3)**2 + (y-0.3)**2))+exp(-20*((x+0.3)**2 + (y+0.3)**2)) + 0.001
    uex = log(rhoex)
    

    if quads:
        mesh = MakeStructured2DMesh(quads=quads, nx=nx, ny=nx, 
                                            mapping=lambda x,y:(xmin+(xmax-xmin)*x, 
                                                                             xmin+(xmax-xmin)*y))
        ELE = QUAD
        ir = IntegrationRule(points = [(0,0), (1,0), (0,1),(1,1)], weights = [1/4, 1/4, 1/4, 1/4] )
        et = "quads"
    else: # unstructured trig mesh
        geo = CSG2d()
        rect = Rectangle( pmin=(-L,-L), pmax=(L,L) )
        geo.Add(rect)
        mesh = Mesh(geo.GenerateMesh(maxh=(xmax-xmin)/nx))
        ELE = TRIG
        et = "trig"
        ir = IntegrationRule(points = [(0,0), (1,0), (0,1)], weights = [1/6, 1/6, 1/6] )
    
    print(mesh.ne)
    V = H1(mesh,order=order)
    fes = V
    uh = GridFunction(fes)
    
    # Used to store data of previous iteration
    uh0 = GridFunction(V)
    rho0 = GridFunction(V)
    # Used in Newton's iterations
    rhok = GridFunction(V)
    rhouk = GridFunction(V)

    rho0.vec[:] = 0
    rhok.vec[:] = 0

    u = fes.TrialFunction()
    v = fes.TestFunction() 


    a = BilinearForm(fes)
    a += m*rho0**m*grad(u)*grad(v)*dx(intrules={ELE:ir})
    a += rhok*u*v/dt*dx(intrules={ELE:ir})

    f = LinearForm(fes)
    f += (rhouk-rhok+rho0)*v/dt*dx(intrules={ELE:ir})

    # initial data: point interpolation
    uh0.Set(uex)

    # Avoid NAN values
    pos = uh0.vec.FV().NumPy() > -1000
    uh0.vec.FV().NumPy()[~pos] = -np.inf
    rho0.vec.FV().NumPy()[:] = np.exp(uh0.vec.FV().NumPy())

    activeDofs = BitArray(fes.ndof)

    def myNewton(damp0=1, tol0 = 1e-8):
        uh.vec.data = uh0.vec
        rhok.vec.FV().NumPy()[:] = np.exp(uh.vec.FV().NumPy()[:])
        # set -inf to zero for simplicity!!! FIXME LATER
        pos = rhok.vec.FV().NumPy()==0
        uh.vec.FV().NumPy()[pos] = 0 
        rhouk.vec.FV().NumPy()[:] = rhok.vec.FV().NumPy()[:]*uh.vec.FV().NumPy()[:]

        count = 0
        ener0 = (uh.vec.FV().NumPy()[:]-1).dot(rhok.vec.FV().NumPy()[:])

        tol = abs(ener0)*tol0 # relative tolerance
        while True:
            count += 1
            # hack zero
            a.Assemble()
            f.Assemble()
            if count==1: # locate active Dofs
                rows,cols,vals = a.mat.COO()
                A = sp.csr_matrix((vals,(rows,cols)))
                active = A.diagonal()>cutOff
                activeDofs[:] = 1
                # THIS IS SUPER SLOW...
                for i in range(fes.ndof):
                    if active[i]==0:
                        activeDofs[i] = 0



            uh.vec.data = a.mat.Inverse(freedofs = activeDofs,inverse="sparsecholesky")*f.vec
            rhok.vec.FV().NumPy()[active] = np.exp(uh.vec.FV().NumPy()[active])
            rhouk.vec.FV().NumPy()[active] = rhok.vec.FV().NumPy()[active]*uh.vec.FV().NumPy()[active]


            ener1 = (uh.vec.FV().NumPy()[active]-1).dot(rhok.vec.FV().NumPy()[active])
            err = abs(ener1-ener0)
            ener0 = ener1
            if err < tol or count == 2:
                uh.vec.FV().NumPy()[~active] = -np.inf
                break
            if np.isnan(err) or count==30:
                print(count,"FAILED")
                stop 

        return count 


    pnts_x = np.linspace(xmin,xmax,nx+1)  
    if plot==True:                
        scene = Draw(rhok, mesh)
    step = 0



    while t.Get() < tend-dt.Get()/2:
        t.Set(t.Get()+dt.Get())
        step += 1

        ct = myNewton()

        uh0.vec.data = uh.vec
        rho0.vec.FV().NumPy()[:] = np.exp(uh0.vec.FV().NumPy())
        if step % 10==0 and plot==True:
            scene.Redraw()
            
    if plot: scene.Redraw()
    
    vtk = VTKOutput(ma=mesh,
                coefs=[rhok],
                names = ["density"],
                filename="logDensityMergingGaussians_m_{}_elements_{}_tstop_{}_".format(m,mesh.ne,tend)+et,
                subdivision=5)
    vtk.Do()
    
    return None

In [3]:
def massinvB(mesh):
    
    vertices={}
    for v in mesh.vertices:
        vertices[v] = v.point
    edges = {}

    for e in mesh.edges:
        edges[e] = e.vertices
        
    angles = {}
    lengths = {}
    for el in mesh.Elements():
        
        lengloc = {}
        for e in el.edges:
                x1 = vertices[edges[e][0]][0]
                y1 = vertices[edges[e][0]][1]
                x2 = vertices[edges[e][1]][0]
                y2 = vertices[edges[e][1]][1]

                dx = (x1-x2)
                dy = (y1-y2)
                l = (np.sqrt((dx)**2+(dy)**2))
                lengths[e] = l
                lengloc[e] = l
            

        for e in el.edges:
            elist = list(lengloc.keys())
            elist.remove(e)
            c = lengths[e]
            a = lengths[elist[0]]
            b = lengths[elist[1]]
            theta = np.arccos((a**2 + b**2 -c**2)/2/a/b)
            try:
                angles[e].append(theta)
            except:
                angles[e] = [theta]


    sortedAngles = dict(sorted(angles.items(),key=lambda t: t[0].nr ))
    invB = []
    B = []
    for key,val in sortedAngles.items():
        l = lengths[key]
        # FIX: boundary nodes need to be multiplied by 2
        entry = (0.5*np.sum(1/np.tan(sortedAngles[key])))
        invB.append(0.5*l*1/entry)
        B.append(0.5*l*entry)

    return invB,B
    
    #return sp.sparse.spdiags(invB, 0, len(invB), len(invB))



In [4]:
def mixed(
    nx = 20,
    m = 2,
    s0 = 3,
    order = 1,
    dt0 =0.01,
    t = Parameter(0),
    tstop = 1,
    plot = 0,
    quads = 0
):
    
    tau = Parameter(dt0)
    dim = 2
    t0 = 1
    # reference solution
    rho_ex = exp(-20*((x-0.3)**2 + (y-0.3)**2))+exp(-20*((x+0.3)**2 + (y+0.3)**2)) + 0.001

    xmin, xmax = -1,1
    L = xmax

    if quads:
        mesh = MakeStructured2DMesh(quads=quads, nx=nx, ny=nx, 
                                            mapping=lambda x,y:(xmin+(xmax-xmin)*x, 
                                                                             xmin+(xmax-xmin)*y))
        et = "quads"
    else: # unstructured trig mesh
        geo = CSG2d()
        rect = Rectangle( pmin=(-L,-L), pmax=(L,L) )
        geo.Add(rect)
        mesh = Mesh(geo.GenerateMesh(maxh=(xmax-xmin)/nx))
        et = "trig"
        
    print(mesh.ne)
    
    V = L2(mesh)   # density
    H = FacetFESpace(mesh) # one dof per edge (for upwinding impl)    
    W = HDiv(mesh) # RT0 velocity
    fes = FESpace([V, W])
    
    (rho, u), (eta, v) = fes.TnT()    
    gfu = GridFunction(fes)
    rhoh, uh = gfu.components
    rhoh0 = GridFunction(V) # previous den
    uh0 = GridFunction(W)   # previous velocity
    rhoh1 = GridFunction(H) # average den
    
    
    a  = BilinearForm(fes)
    n = specialcf.normal(mesh.dim)

    # linearized bilinear form
    #flux = u*n*rhoh1 # average flux instead of upwind
    flux = u*n*IfPos(uh*n, rhoh0, 2*rhoh1-rhoh0)
    
    a += (rho-rhoh0)/tau*eta*dx
    a += flux*eta*dx(element_boundary=True)
    # vel
    if not quads:
        gf1 = GridFunction(FacetFESpace(mesh))
        temp,temp1 = massinvB(mesh)
        for i in range(len(gf1.vec)):
            gf1.vec[i] = temp1[i]
        a += gf1*u*n*v*n*dx(element_boundary=True)
    else:
        ir0 = IntegrationRule(points = [(0,0.5), (1,0.5)], weights = [1/2, 1/2] )
        ir1 = IntegrationRule(points = [(0.5,0), (0.5,1)], weights = [1/2, 1/2] )
        a += u[0]*v[0]*dx(intrules={QUAD: ir0})
        a += u[1]*v[1]*dx(intrules={QUAD: ir1})
    
    # nonlinear pres
    p = m/(m-1)*rho**(m-1)
    # linearize pres
    #p = m/(m-1)*rhoh0**(m-1)+ m*rhoh0**(m-2)*(rho-rhoh0)
    a += -p*div(v)*dx 

    # hack a DG average (for Newton!)
    phi, psi = H.TnT()
    af = BilinearForm(H)
    af += phi*psi*dx(element_boundary=True)
    af.Assemble()
    invaf = af.mat.CreateSmoother(H.FreeDofs())
    ff = LinearForm(H)
    ff += rhoh*psi*dx(element_boundary=True)
    
    iteration = 0
    # initial data: L2-projection
    rhoh.Set(rho_ex)

    if plot == 1:
        scene = Draw(rhoh, mesh)
    with TaskManager():
        while t.Get() < tstop - tau.Get()/2:
            dt = tau.Get()
            t.Set(t.Get()+dt)                
            rhoh0.vec.data = rhoh.vec
            uh0.vec.data = uh.vec
            ff.Assemble()
            rhoh1.vec.data = invaf*ff.vec
            
            # The linearized scheme only need 1 Newton iteration 
            it = Newton(a, gfu, printing=False, maxit=10)          
            iteration += 1
            if plot == 1 and iteration % 100 == 0:
                scene.Redraw()  
    if plot: scene.Redraw()  
    vtk = VTKOutput(ma=mesh,
                coefs=[rhoh],
                names = ["density"],
                filename="mixedMergingGaussians_m_{}_elements_{}_tstop_{}_".format(m,mesh.ne,tstop)+et,
                subdivision=5)
    # Exporting the results:
    vtk.Do()
    return None


In [5]:
m=4
nx = 20
dt = 1e-3
logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 0.001,nx = nx, plot=1,quads=0)
#logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 0.5,nx = nx, plot=0,quads=0)
#logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 1,nx = nx, plot=0,quads=0)

904


WebGuiWidget(value={'ngsolve_version': '6.2.2105-211-g63bbcb022', 'mesh_dim': 2, 'order2d': 2, 'order3d': 2, '…

In [61]:
m=4
nx = 30
dt = 1e-3
logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 0.001,nx = nx, plot=0,quads=1)
#logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 0.5,nx = nx, plot=0,quads=1)
#logDensity(m=m,dt0=dt,t = Parameter(0),tstop = 1,nx = nx, plot=0,quads=1)

900


In [63]:
m=4
nx = 20
dt = 1e-3
#mixed(m=m,dt0=dt,t = Parameter(0),tstop = 0.001,nx = nx, plot=1,quads=0)
mixed(m=m,dt0=dt,t = Parameter(0),tstop = 0.5,nx = nx, plot=0,quads=0)
mixed(m=m,dt0=dt,t = Parameter(0),tstop = 1,nx = nx, plot=0,quads=0)

904


WebGuiWidget(value={'ngsolve_version': '6.2.2105-211-g63bbcb022', 'mesh_dim': 2, 'order2d': 2, 'order3d': 2, '…

904


WebGuiWidget(value={'ngsolve_version': '6.2.2105-211-g63bbcb022', 'mesh_dim': 2, 'order2d': 2, 'order3d': 2, '…

In [64]:
m=4
nx = 30
dt = 1e-3
mixed(m=m,dt0=dt,t = Parameter(0),tstop = 0.001,nx = nx, plot=0,quads=1)
mixed(m=m,dt0=dt,t = Parameter(0),tstop = 0.5,nx = nx, plot=0,quads=1)
mixed(m=m,dt0=dt,t = Parameter(0),tstop = 1,nx = nx, plot=0,quads=1)

900
900
900
